In [1]:
from llama_index.core.indices.multi_modal.base import MultiModalVectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import SimpleDirectoryReader, StorageContext
from llama_index.embeddings.clip import ClipEmbedding
from llama_index.embeddings.cohere import CohereEmbedding

import chromadb
import os
from dotenv import load_dotenv

Tutorial: https://docs.llamaindex.ai/en/stable/examples/multi_modal/ollama_cookbook/#multi-modal-rag

In [2]:
_ = load_dotenv(".env")

In [3]:
# Create Cohere embedding 
embed_model = CohereEmbedding(
    api_key=os.getenv('COHERE_API_KEY')
)

In [4]:
image_embed_model = ClipEmbedding()

In [5]:

# Create a local Qdrant vector store
chroma_client = chromadb.EphemeralClient()


In [6]:
# Create a multimodal collection
text_collection = chroma_client.get_or_create_collection(
    name="text_collection", 
)

In [7]:
# Create a multimodal collection
image_collection = chroma_client.get_or_create_collection(
    name="image_collection", 
)

In [8]:
text_store = ChromaVectorStore(chroma_collection=text_collection)


In [9]:
image_store = ChromaVectorStore(chroma_collection=image_collection)

In [10]:
storage_context = StorageContext.from_defaults(
    vector_store=text_store, image_store=image_store
)

In [11]:
# Create the MultiModal index
documents = SimpleDirectoryReader("../data").load_data()
index = MultiModalVectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    image_embed_model=image_embed_model,
    embed_model = embed_model  

)

In [13]:
from llama_index.core.prompts import PromptTemplate
from llama_index.core.query_engine import SimpleMultiModalQueryEngine
from llama_index.multi_modal_llms.ollama import OllamaMultiModal

qa_tmpl_str = (
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "answer the query.\n"
    "Query: {query_str}\n"
    "Answer: "
)
qa_tmpl = PromptTemplate(qa_tmpl_str)

mm_model = OllamaMultiModal(model="llava:7b", request_timeout=60.0)

query_engine = index.as_query_engine(llm=mm_model, text_qa_template=qa_tmpl)


In [14]:
#prueba 1
query_str = "Tell me more about the Cowboy Boots"
response = query_engine.query(query_str)
print(response)

 Based on the image provided, these cowboy boots are designed with a denim material, giving them a fashionable and original look. The boots are characterized by their high heels, which adds to their distinctive style. The design includes floral details in pink and white hues, adding an extra layer of visual interest. These boots appear to be part of a collection or brand called "Original Denim," as indicated by the text on the image. Additionally, there is a social media handle "@ESTELA.MADARIAGA.LASALA" which suggests that this could be the designer or the brand behind these unique cowboy boots. 


In [15]:
#prueba 2
query_str = "Tell me who is the designer of the Cowboy Boots"
response = query_engine.query(query_str)
print(response)

 The designer of the cowboy boots in the image is "Original Denim", as indicated by the text "@ESTELA.MADARIAGA.LASALA". 


In [18]:
#prueba 3
query_str = "shoes made in cocodrille leather"
response = query_engine.query(query_str)
print(response)

 The image you've provided shows a pair of shoes with a pink and purple color scheme, featuring a distinctive design element on the sole. While the shoes are not made of crocodile leather, they appear to have a similar texture or material on their soles, which could give a similar appearance to those made in cocodrille leather. However, the brand name "Stella Madariaga" is clearly visible across the side of one shoe, indicating that these shoes are not made by this specific brand. To confirm if the shoes are indeed made of crocodile leather, additional information or confirmation from the manufacturer would be required. 


In [ ]:
# prueba 4: 
retrival_result = index.as_retriever( ).retrieve("Sneakers")
retrival_result

[NodeWithScore(node=TextNode(id_='29ab8e88-ac24-4b6b-b449-bb55b29bbbaf', embedding=None, metadata={'file_path': 'd:\\workspace\\proyecto_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\data\\5.txt', 'file_name': '5.txt', 'file_type': 'text/plain', 'file_size': 901, 'creation_date': '2024-11-13', 'last_modified_date': '2024-11-13'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='0340b15d-ae79-4f35-85c0-411790a0c0f5', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'd:\\workspace\\proyecto_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\data\\5.txt', 'file_name': '5.txt', 'file_type': 'text/plain', 'file_size': 901, 'creation_date': '2024-11-13', 'last_modified_date': '2024-11-13'}, hash='c6a6

In [23]:
# prueba 5: 
retrival_result = index.as_retriever( ).retrieve("Sandal")
retrival_result

[NodeWithScore(node=TextNode(id_='3fc4433d-8ece-4246-82c4-ca29f134fe79', embedding=None, metadata={'file_path': 'd:\\workspace\\proyecto_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\data\\14.txt', 'file_name': '14.txt', 'file_type': 'text/plain', 'file_size': 1047, 'creation_date': '2024-11-13', 'last_modified_date': '2024-11-13'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='f79b9d53-6d2e-4f03-bda9-57236cbbb7f6', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'd:\\workspace\\proyecto_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\data\\14.txt', 'file_name': '14.txt', 'file_type': 'text/plain', 'file_size': 1047, 'creation_date': '2024-11-13', 'last_modified_date': '2024-11-13'}, hash

In [ ]:
# prueba 6: 
retrival_result = index.as_retriever( ).text_to_image_retrieve("Sneakers")
retrival_result

[NodeWithScore(node=ImageNode(id_='79423c3a-bc54-4c2f-8511-9b94e9401b65', embedding=None, metadata={'file_path': 'd:\\workspace\\proyecto_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\data\\5.jpg', 'file_name': '5.jpg', 'file_type': 'image/jpeg', 'file_size': 67309, 'creation_date': '2024-11-12', 'last_modified_date': '2024-10-26'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='282f6bdf-8ace-45e9-9c4b-07fdc2629538', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'd:\\workspace\\proyecto_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\data\\5.jpg', 'file_name': '5.jpg', 'file_type': 'image/jpeg', 'file_size': 67309, 'creation_date': '2024-11-12', 'last_modified_date': '2024-10-26'}, hash=

In [ ]:
# prueba 7: 
retrival_result = index.as_retriever( ).text_to_image_retrieve("Sandals")
retrival_result

[NodeWithScore(node=ImageNode(id_='79423c3a-bc54-4c2f-8511-9b94e9401b65', embedding=None, metadata={'file_path': 'd:\\workspace\\proyecto_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\data\\5.jpg', 'file_name': '5.jpg', 'file_type': 'image/jpeg', 'file_size': 67309, 'creation_date': '2024-11-12', 'last_modified_date': '2024-10-26'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='282f6bdf-8ace-45e9-9c4b-07fdc2629538', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'd:\\workspace\\proyecto_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\data\\5.jpg', 'file_name': '5.jpg', 'file_type': 'image/jpeg', 'file_size': 67309, 'creation_date': '2024-11-12', 'last_modified_date': '2024-10-26'}, hash=

In [24]:
# prueba 7: 
retrival_result = index.as_retriever( ).image_to_image_retrieve('../tests/images/12.jpg')
retrival_result

[NodeWithScore(node=ImageNode(id_='5b502227-e3d8-4a16-86c9-29f6c8d893d3', embedding=None, metadata={'file_path': 'd:\\workspace\\proyecto_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\data\\14.jpg', 'file_name': '14.jpg', 'file_type': 'image/jpeg', 'file_size': 171654, 'creation_date': '2024-11-12', 'last_modified_date': '2024-10-26'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='d6929f5d-7357-4f19-97c6-9aaf5042ce25', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'd:\\workspace\\proyecto_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\data\\14.jpg', 'file_name': '14.jpg', 'file_type': 'image/jpeg', 'file_size': 171654, 'creation_date': '2024-11-12', 'last_modified_date': '2024-10-26'},